# Imputation Model Training

Trains and serialises an imputation pipeline to fill missing environmental variables in incoming farm profiles.

## Approach: IterativeImputer + Random Forest

Uses scikit-learn `IterativeImputer` with `RandomForestRegressor` as the base estimator.
Each feature is modelled as a function of all others and the process iterates to convergence.

Random Forest is preferred over KNN here because:
- It handles non-linear relationships and feature interactions naturally
- It does not require feature scaling (no `StandardScaler` needed)
- It is generally more robust on tabular environmental data

## Features

| Column | Type | Notes |
|---|---|---|
| `elevation_m` | Numeric | |
| `slope` | Numeric | |
| `temperature_celsius` | Numeric | |
| `rainfall_mm` | Numeric | |
| `ph` | Numeric | |

`soil_texture_id` is excluded — it is a database FK with no numeric meaning and must always be supplied from soil classification data.

Base features always available from farm geometry: `latitude`, `longitude`, `area_ha`, `coastal`, `riparian`.

## Dataset

`backend/src/scripts/data/farm_master.csv` — 3200 rows, no missing values.
Since there are no real missing values, evaluation uses MCAR masking (missing completely at random)
with 5-fold cross-validation to measure how well the pipeline recovers masked values.

## 1. Imports

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
DATA_PATH = Path("../../backend/src/scripts/data/farm_master.csv")
MODELS_PATH = Path("../src/models/imputation")
MODELS_PATH.mkdir(parents=True, exist_ok=True)

## 2. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Nulls: {df.isnull().sum().sum()}")
print()
print(df.dtypes)
df.head()

## 3. Prepare Features

In [ ]:
# Base features — always available from farm geometry
BASE_FEATURES = ["latitude", "longitude", "area_ha", "coastal", "riparian"]

# Target features — numeric environmental variables that may be missing
# soil_texture_id is excluded: it is a DB FK with no numeric meaning and
# must always be supplied from soil classification data, not imputed.
TARGET_FEATURES = ["elevation_m", "slope", "temperature_celsius", "rainfall_mm", "ph"]
ALL_FEATURES = BASE_FEATURES + TARGET_FEATURES

# Prepare feature matrix — convert booleans to int for sklearn
X_full = df[ALL_FEATURES].copy()
X_full["coastal"] = X_full["coastal"].astype(int)
X_full["riparian"] = X_full["riparian"].astype(int)

print(f"Feature matrix shape: {X_full.shape}")
X_full.head()

## 4. Fit Imputation Pipeline on Full Dataset

In [ ]:
# RandomForestRegressor does not require feature scaling
final_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED),
    max_iter=10,
    random_state=RANDOM_SEED,
)
final_imputer.fit(X_full)
print("Imputer fitted on full dataset.")

## 5. Evaluate Pipeline

Since the dataset has no real missing values, evaluation uses **MCAR masking** (missing completely at random)
with **5-fold cross-validation**. In each fold:
1. Fit the imputer on the training split
2. Mask 30% of target values in the test split
3. Impute and measure recovery in original units

Metrics are averaged across all folds.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
MASK_RATE = 0.30
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

fold_results = {col: [] for col in TARGET_FEATURES}

for fold, (train_idx, test_idx) in enumerate(kf.split(X_full), 1):
    X_train = X_full.iloc[train_idx]
    X_test = X_full.iloc[test_idx].copy()

    imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED),
        max_iter=10,
        random_state=RANDOM_SEED,
    )
    imputer.fit(X_train)

    true_values = {}
    for col in TARGET_FEATURES:
        mask = rng.random(len(X_test)) < MASK_RATE
        true_values[col] = (X_test.loc[X_test.index[mask], col].values.copy(), mask)
        X_test.loc[X_test.index[mask], col] = np.nan

    # RF imputer outputs original-scale values directly — no inverse_transform needed
    X_imp = imputer.transform(X_test)
    X_imp_df = pd.DataFrame(X_imp, columns=X_full.columns, index=X_test.index)

    for col in TARGET_FEATURES:
        y_true, mask = true_values[col]
        y_pred = X_imp_df.loc[X_test.index[mask], col].values
        if len(y_true) == 0:
            continue
        fold_results[col].append(np.sqrt(mean_squared_error(y_true, y_pred)))

    print(f"Fold {fold}/{N_SPLITS} done")

In [ ]:
# Suggested RMSE thresholds based on domain context and prior model performance
THRESHOLDS = {
    "elevation_m":         100.0,  # metres  — GEE SRTM MAE ~11m; 100m reflects fallback tolerance
    "slope":                 5.0,  # degrees — acceptable planting slope error
    "temperature_celsius":   2.0,  # °C      — agronomic species sensitivity
    "rainfall_mm":         200.0,  # mm      — species rainfall band width
    "ph":                    1.0,  # pH unit — typical soil pH tolerance range
}

print("=" * 72)
print(f"EVALUATION SUMMARY — {N_SPLITS}-fold CV, {MASK_RATE:.0%} MCAR masking")
print("=" * 72)
header = f"{'Feature':25s} | {'RMSE (mean±std)':>18} | {'Threshold':>10} | {'Status'}"
print(header)
print("-" * 72)
for col in TARGET_FEATURES:
    scores = fold_results[col]
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    threshold = THRESHOLDS[col]
    status = "PASS" if mean_score <= threshold else "FAIL"
    rmse_str = f"{mean_score:.3f} ± {std_score:.3f}"
    print(f"{col:25s} | {rmse_str:>18} | {threshold:>10.1f} | {status}")

## 6. Save Model Artefacts

> **Only run this cell once the model logic above has been reviewed and approved.**
> Serialises the fitted imputer and metadata to `src/models/imputation/`.

In [ ]:
joblib.dump(final_imputer, MODELS_PATH / "imputation_pipeline.joblib")
joblib.dump(list(X_full.columns), MODELS_PATH / "feature_columns.joblib")

print("Saved:")
for f in sorted(MODELS_PATH.glob("*.joblib")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:6.1f} KB")

## 7. Usage Example

How the imputation service uses the saved imputer:

```python
import joblib
import numpy as np
import pandas as pd

imputer = joblib.load("src/models/imputation/imputation_pipeline.joblib")
feature_columns = joblib.load("src/models/imputation/feature_columns.joblib")

# Incoming farm profile with some missing numeric values
# Note: soil_texture_id must always be supplied separately — it is not imputed
farm = {
    "latitude": -8.57, "longitude": 126.68, "area_ha": 1.2,
    "coastal": 0, "riparian": 0,
    "elevation_m": np.nan, "slope": np.nan,
    "temperature_celsius": 24.0, "rainfall_mm": np.nan,
    "ph": 6.5,
}

X = pd.DataFrame([farm])[feature_columns]
X_imputed = imputer.transform(X)
result = dict(zip(feature_columns, X_imputed[0]))
print(result)
```